# PROMPT ENGINEERING

Understanding


*   Chain of Thought
*   Self Consistency
*   Tree of Thought
*   Group of Thought






In [ ]:
!pip -q install transformers accelerate sentencepiece

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

In [ ]:
def ask_llm(prompt):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    return answer.strip()

In [ ]:
problem = """
You have three boxes. One contains only apples, one contains only oranges,
and one contains a mix of both.

Each box has a label, but you are told that ALL three labels are wrong.

You are allowed to pick exactly ONE fruit from exactly ONE box
(without seeing inside) to correctly relabel all three boxes.

Which box should you pick from, and what is the correct label
for each of the three boxes? Explain your reasoning.
"""

print(problem)

**1 Baseline Prompt**

In [ ]:
print("="*60)
print("BASELINE PROMPT")
print("="*60)

prompt = problem

print(ask_llm(prompt))

**No Explaination**

In [ ]:
print("="*60)
print("BASELINE PROMPT")
print("="*60)

prompt = f"""
Answer only.

{problem}
"""

print(ask_llm(prompt))

**2 Zero-Shot Prompting**

In [ ]:
print("="*60)
print("ZERO SHOT PROMPTING")
print("="*60)

prompt = f"""
You are an expert mathematician.

Solve the following question.

Question

{problem}

Answer
"""

print(ask_llm(prompt))

**3 Better Zero Shot**

In [ ]:
print("="*60)
print("ZERO SHOT (STRICT)")
print("="*60)

prompt = f"""
You are an expert math solver.

Read carefully.

Return ONLY the final answer.

Question

{problem}

Final Answer
"""

print(ask_llm(prompt))

**4 Few Shot Prompting**

In [ ]:
print("="*60)
print("FEW SHOT")
print("="*60)

prompt = """
Question:
Ali has 2 apples and 3 oranges.

How many fruits he have in basket?

Answer:
5

Question:
Sara has 10 chocolates.

She eats 4.

Answer:
6

Question:
You have three boxes. One contains only apples, one contains only oranges,
and one contains a mix of both. Each box has a label, but all three labels
are wrong. You may pick exactly ONE fruit from exactly ONE box to correctly
relabel all three boxes. Which box should you pick from, and what is the
correct label for each box?
Answer:
"""

print(ask_llm(prompt))

**5 Chain of Thought**

In [ ]:
print("="*60)
print("CHAIN OF THOUGHT")
print("="*60)

prompt = f"""
Solve the problem.

Think step by step.

Question

{problem}

Answer
"""

print(ask_llm(prompt))

**6 Zero-Shot CoT**

This is the famous paper prompt.

Only one sentence changes.

In [ ]:
print("="*60)
print("ZERO SHOT CoT")
print("="*60)

prompt = f"""
{problem}

Let's think step by step.
"""

print(ask_llm(prompt))

**7 Self Consistency**

In [ ]:
def ask_sample(prompt):

    messages = [
        {
            "role":"user",
            "content":prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.8,
        top_p=0.9
    )

    return tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

In [ ]:
prompt = f"""
{problem}

Let's think step by step.

At the end, write your answer in exactly this format:

Final Answer: <answer>
"""

for i in range(5):

    print("="*50)
    print("Reasoning Path", i+1)
    print("="*50)

    print(ask_sample(prompt))

In [ ]:
from collections import Counter

answers = []

for i in range(5):

    print("="*50)
    print("Reasoning Path", i+1)
    print("="*50)

    response = ask_sample(prompt)
    print(response)

    answers.append(response)

In [ ]:
import re
from collections import Counter

print("\n" + "="*60)
print("FINAL ANSWERS FROM EACH PATH")
print("="*60)

final_answers = []

for i, response in enumerate(answers, 1):

    # Look for "Final Answer: ..."
    match = re.search(
        r"Final\s*Answer\s*:\s*(.+)",
        response,
        flags=re.IGNORECASE
    )

    if match:
        answer = match.group(1).strip()

    else:
        # Fallback: extract the last number if the expected format is missing
        numbers = re.findall(r"\d+(?:\.\d+)?", response)

        if numbers:
            answer = numbers[-1]
        else:
            answer = "Unknown"

    final_answers.append(answer)

    print(f"Path {i}: {answer}")

# ----------------------------
# Majority Voting
# ----------------------------

vote = Counter(final_answers)

most_common_answer, votes = vote.most_common(1)[0]

print("\n" + "="*60)
print("MAJORITY VOTING")
print("="*60)

for ans, count in vote.items():
    print(f"{ans} : {count} vote(s)")

print("\n" + "-"*60)
print(f"Final Selected Answer : {most_common_answer}")
print(f"Majority Votes        : {votes}/{len(final_answers)}")
print("-"*60)

In [ ]:
import re
from collections import Counter

print("\n" + "="*60)
print("FINAL ANSWERS FROM EACH PATH")
print("="*60)

final_answers = []

for i, response in enumerate(answers, 1):

    # ----------------------------------------
    # First try to extract "Final Answer:"
    # ----------------------------------------
    match = re.search(
        r"Final\s*Answer\s*:\s*(.+)",
        response,
        flags=re.IGNORECASE
    )

    if match:
        raw_answer = match.group(1).strip()

        # Extract the FIRST number after "Final Answer:"
        num = re.search(r"\d+(?:\.\d+)?", raw_answer)

        if num:
            answer = num.group(0)
        else:
            answer = raw_answer

    else:
        # ----------------------------------------
        # Fallback:
        # No "Final Answer:" found.
        # Take the LAST number from the entire response.
        # ----------------------------------------
        numbers = re.findall(r"\d+(?:\.\d+)?", response)

        if numbers:
            answer = numbers[-1]
        else:
            answer = "Unknown"

    final_answers.append(answer)

    print(f"Path {i}: {answer}")

# ----------------------------------------
# Majority Voting
# ----------------------------------------

vote = Counter(final_answers)

most_common_answer, votes = vote.most_common(1)[0]

print("\n" + "="*60)
print("MAJORITY VOTING")
print("="*60)

for ans, count in vote.items():
    print(f"{ans} : {count} vote(s)")

print("\n" + "-"*60)
print(f"Final Selected Answer : {most_common_answer}")
print(f"Majority Votes        : {votes}/{len(final_answers)}")
print("-"*60)

**Tree of Thoughts (ToT)**

In [ ]:
def generate_thought(problem):

    prompt = f"""
You are solving a reasoning problem using the Tree of Thoughts (ToT) approach.

Your task is to generate ONE possible reasoning path.

Instructions:
- Think step by step.
- Explore one possible solution.
- Do not assume this is the only correct reasoning.
- This reasoning will later be compared with other candidate thoughts.
- End with:

Final Answer: <your answer>

Problem:
{problem}
"""

    return ask_llm(prompt)

In [ ]:
def generate_multiple_thoughts(problem, n=3):

    thoughts = []

    for i in range(n):

        prompt = f"""
You are solving a reasoning problem using the Tree of Thoughts (ToT) approach.

Generate ONE independent reasoning path.

Instructions:
- Think step by step.
- Explore a possible solution.
- This is only ONE candidate thought.
- Different reasoning paths will be generated and compared later.
- Focus on logical and consistent reasoning.
- End with:

Final Answer: <your answer>

Problem:
{problem}
"""

        messages = [
            {
                "role": "user",
                "content": prompt
            }
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(text, return_tensors="pt").to(model.device)

        outputs = model.generate(
            **inputs,
            do_sample=True,
            temperature=0.9,
            top_p=0.9,
            max_new_tokens=250
        )

        answer = tokenizer.decode(
            outputs[0][inputs.input_ids.shape[1]:],
            skip_special_tokens=True
        )

        thoughts.append(answer)

    return thoughts

In [ ]:
thoughts = generate_multiple_thoughts(problem, n=3)

print("=" * 80)
print("TREE OF THOUGHTS : GENERATED CANDIDATE REASONING PATHS")
print("=" * 80)

for i, thought in enumerate(thoughts, start=1):

    print(f"\n{'=' * 80}")
    print(f"THOUGHT PATH {i}")
    print(f"{'=' * 80}")

    print(thought)

**Graph of Thoughts (GoT)**

In [ ]:
def initial_thoughts(problem):

    prompt = f"""
You are working in a Group of Thoughts (GoT) reasoning framework.

Your task is to generate EXACTLY TWO different high-level solution ideas.

IMPORTANT RULES:
- Do NOT solve the problem.
- Do NOT calculate the final answer.
- Do NOT provide step-by-step reasoning.
- Each idea should describe only a possible approach.
- The two ideas should use different reasoning strategies.

Problem:
{problem}

Return ONLY in the following format:

Idea 1:
<one or two sentences describing the first approach>

Idea 2:
<one or two sentences describing the second approach>
"""

    return ask_llm(prompt)

In [ ]:
def expand_idea(problem, idea):

    prompt = f"""
You are working in a Group of Thoughts (GoT) reasoning framework.

Expand the given solution idea into a detailed reasoning path.

IMPORTANT RULES:
- Expand ONLY the given idea.
- Do NOT introduce a completely new approach.
- Keep the reasoning consistent with the original idea.
- Explain the reasoning step by step.
- Do NOT provide the final answer.
- Do NOT conclude the problem.

Problem:
{problem}

Solution Idea:
{idea}

Return ONLY in the following format:

Expanded Reasoning:
<step-by-step reasoning based only on the given idea>
"""

    return ask_llm(prompt)

In [ ]:
def merge_ideas(problem, idea1, idea2):

    prompt = f"""
You are working in a Group of Thoughts (GoT) reasoning framework.

Your task is to combine the two reasoning paths into ONE stronger reasoning path.

IMPORTANT RULES:
- Do NOT choose one idea and ignore the other.
- Identify the strengths of BOTH reasoning paths.
- Merge their useful information into one coherent reasoning.
- Remove any redundant or contradictory statements.
- Preserve the logical flow.
- Do NOT generate a completely new approach.
- Do NOT provide the final answer yet.

Problem:
{problem}

Expanded Reasoning 1:
{idea1}

Expanded Reasoning 2:
{idea2}

Return ONLY in the following format:

Merged Reasoning:
<one improved reasoning path that combines both ideas>
"""

    return ask_llm(prompt)

In [ ]:
def final_answer(problem, merged):

    prompt = f"""
You are working in a Group of Thoughts (GoT) reasoning framework.

The reasoning from multiple solution ideas has already been generated,
expanded, and merged into one improved reasoning path.

Your task is to:

1. Use ONLY the merged reasoning below.
2. Solve the problem using that reasoning.
3. Do NOT generate a new approach.
4. Do NOT ignore the merged reasoning.
5. Clearly explain the final reasoning.
6. Provide the final answer.

Problem:
{problem}

Merged Reasoning:
{merged}

Return ONLY in the following format:

Final Reasoning:
<brief reasoning based on the merged reasoning>

Final Answer:
<answer only>
"""

    return ask_llm(prompt)

In [ ]:
ideas = initial_thoughts(problem)

print("\n" + "=" * 80)
print("GROUP OF THOUGHTS (GoT) : INITIAL IDEA GENERATION")
print("=" * 80)

print("\nThe model generated multiple high-level solution ideas.")
print("These are NOT complete solutions.")
print("They represent different approaches that will be expanded and merged later.\n")

print("=" * 80)
print("GENERATED INITIAL IDEAS")
print("=" * 80)

print(ideas)

print("\n" + "=" * 80)
print("NEXT STEP")
print("=" * 80)

print("Each idea will now be expanded into a detailed reasoning path.")